# JEPA projector: clean retrain (fixed tokenizer)

**Why this notebook exists.** The first projector was trained with `AutoTokenizer`, which on transformers 5.0.0 silently returns a broken Llama tokenizer for Vicuna (spaces become `<unk>`, words lose their `▁` prefix). This notebook trains the same model (frozen I-JEPA ViT-H/16 -> trainable MLP 1280-4096-4096 -> frozen 4-bit Vicuna-7B-v1.5) with `LlamaTokenizer` and a hard assert on the tokenizer output.

**Changes vs. the old script (everything else is identical):** correct tokenizer + assert; pad tokens and BOS are masked out of the loss; an EOS token is appended to every caption; grad-clip is applied once per optimizer step (not per micro-batch); LR warmup + cosine decay; non-finite batches are skipped and weights rolled back if they ever go NaN; seeded, resumable data order.

**Setup (Kaggle):** Accelerator = GPU T4 x2, Internet = On. Add inputs: the COCO-2014 dataset (`nadaibrahim/coco2014`) and the dataset that contains your `manifest.json`.

**Run order:** cell 1 (install) -> restart the session -> cells 2-8. `SANITY = False` here, so this is the full 3-epoch run. Set it to `True` for a 100-step check.

## 1. Install the pinned stack, then restart the session

In [1]:
!pip install -q "transformers==5.16.1" "accelerate==1.14.0" "bitsandbytes==0.50.2" "pillow==12.3.0" sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 50.2 MB/s eta 0:00:00


## 2. Config and environment check

In [2]:
import os, json, glob, math, time, random, hashlib, shutil
import numpy as np, torch, torch.nn as nn
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import transformers, accelerate, bitsandbytes, PIL

# ------------------------- CONFIG -------------------------
SANITY      = False                  # True: ~100 optimizer steps on 1,600 images. False: full 3-epoch run.
MANIFEST    = None                  # path to manifest.json; None = the manifest-data-split dataset
LLM_PATH    = "lmsys/vicuna-7b-v1.5"
VIT_ID      = "facebook/ijepa_vith16_1k"
COCO_ROOT   = "/kaggle/input/datasets/nadaibrahim/coco2014"
IMG_DIR     = f"{COCO_ROOT}/val2014/val2014"
CAP_JSON    = f"{COCO_ROOT}/captions/annotations/captions_val2014.json"
OUT_DIR     = "/kaggle/working/ckpt_fixed"

BATCH, ACCUM = 2, 8                 # same as before: effective batch 16
LR, WD, CLIP = 2e-4, 0.0, 1.0
EPOCHS      = 1 if SANITY else 3
N_TRAIN     = 1600 if SANITY else None
WARMUP_OPT  = 10 if SANITY else 40  # warmup length in optimizer steps
VAL_EVERY   = 25 if SANITY else 50  # in optimizer steps
SAVE_EVERY  = 25 if SANITY else 50  # latest.pt (full state), in optimizer steps
MILESTONE_EVERY = 100               # weights-only snapshot every N optimizer steps
PUSH        = True                  # push checkpoints to a NEW private Kaggle dataset under your account
DATASET_SLUG = "jepa-projector-fixed-ckpts"   # becomes <your-username>/jepa-projector-fixed-ckpts
PUSH_EVERY  = 100                   # optimizer steps between pushes (multiple of MILESTONE_EVERY); a final push always happens
MAX_BAD_STREAK  = 30                # abort if this many consecutive batches are non-finite
NUM_WORKERS = 2                     # set 0 if DataLoader workers cause trouble
RESUME_FROM = None                  # path to a latest.pt written by THIS notebook (auto-resumes from OUT_DIR if present)
WARM_START_FROM = None              # optional: weights-only init from an old checkpoint (experiment; see notes)
SEED        = 0
# ----------------------------------------------------------

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
PINNED = dict(transformers="5.16.1", accelerate="1.14.0", bitsandbytes="0.50.2", PIL="12.3.0")
have = dict(transformers=transformers.__version__, accelerate=accelerate.__version__,
            bitsandbytes=bitsandbytes.__version__, PIL=PIL.__version__)
print("versions:", have, "| torch", torch.__version__)
mismatch = {k: (have[k], v) for k, v in PINNED.items() if have[k] != v}
if mismatch: print("WARNING version mismatch (have, wanted):", mismatch, "- the tokenizer assert below is the real safety net")
assert torch.cuda.is_available(), "No GPU: set Accelerator to T4 x2 and restart"
device = "cuda:0"
print("GPUs:", torch.cuda.device_count(), torch.cuda.get_device_name(0))
os.makedirs(OUT_DIR, exist_ok=True)

versions: {'transformers': '5.16.1', 'accelerate': '1.14.0', 'bitsandbytes': '0.50.2', 'PIL': '12.3.0'} | torch 2.11.0+cu128
GPUs: 2 Tesla T4


## 3. Data (same manifest, same first-caption rule as before)

In [3]:
MANIFEST_PATH = MANIFEST or "/kaggle/input/datasets/ponishbhatia2/manifest-data-split/manifest.json"
assert os.path.exists(MANIFEST_PATH), f"manifest.json not found at {MANIFEST_PATH}. Attach it as a dataset input or set MANIFEST."
manifest = json.load(open(MANIFEST_PATH))
print("manifest:", MANIFEST_PATH, {k: len(v) for k, v in manifest.items()})
for k in ("projector_train", "tuned_lens_val", "tuned_lens_test"):
    assert k in manifest, f"manifest is missing key '{k}'"
val_names = manifest["tuned_lens_val"] + manifest["tuned_lens_test"]
assert not (set(manifest["projector_train"]) & set(val_names)), "train/val overlap in manifest"

cap_raw = json.load(open(CAP_JSON))
id2f = {im["id"]: im["file_name"] for im in cap_raw["images"]}
first_cap = {}
for a in cap_raw["annotations"]:
    first_cap.setdefault(id2f[a["image_id"]], a["caption"])      # first caption in file order, as in the old training
del cap_raw

train_all  = [{"image": f, "caption": first_cap[f]} for f in manifest["projector_train"] if f in first_cap]
val_recs   = [{"image": f, "caption": first_cap[f]} for f in val_names if f in first_cap]
train_recs = train_all[:N_TRAIN] if N_TRAIN else train_all
assert all(os.path.exists(f"{IMG_DIR}/{r['image']}") for r in (train_recs[:20] + val_recs[:20])), f"images not found under {IMG_DIR}"
vh = hashlib.sha1(json.dumps(val_recs, sort_keys=True).encode()).hexdigest()[:10]
print(f"train {len(train_recs)} (of {len(train_all)}) | val {len(val_recs)} | val hash {vh} (old training notebook: f839b87aae)")
print("example:", train_recs[0])

manifest: /kaggle/input/datasets/ponishbhatia2/manifest-data-split/manifest.json {'pope': 500, 'tuned_lens_train': 350, 'tuned_lens_val': 75, 'tuned_lens_test': 75, 'projector_train': 5000}
train 5000 (of 5000) | val 150 | val hash f839b87aae (old training notebook: f839b87aae)
example: {'image': 'COCO_val2014_000000351335.jpg', 'caption': 'A man holding a cell phone sitting in a yellow seat.'}


## 4. Tokenizer: explicit `LlamaTokenizer` + hard assert

In [4]:
from transformers import LlamaTokenizer
tokenizer = LlamaTokenizer.from_pretrained(LLM_PATH)
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

test = "A yellow clock that is on the floor."
toks = tokenizer.convert_ids_to_tokens(tokenizer(test).input_ids)
EXPECT = ["<s>", "▁A", "▁yellow", "▁clock", "▁that", "▁is", "▁on", "▁the", "▁floor", "."]
print(type(tokenizer).__name__, toks)
assert toks == EXPECT, f"TOKENIZER BROKEN, do not train: {toks}"

lens, n_unk = [], 0
for r in train_recs + val_recs:
    ids = tokenizer(r["caption"].strip()).input_ids
    lens.append(len(ids)); n_unk += ids.count(tokenizer.unk_token_id)
print(f"captions: {len(lens)} | token length mean {np.mean(lens):.1f} max {max(lens)} | <unk> tokens: {n_unk}")
assert n_unk == 0, "captions contain <unk> tokens: investigate before training"
MAXLEN = 64                                   # BOS + caption + EOS, truncated
print("fraction truncated at", MAXLEN, ":", float(np.mean(np.array(lens) + 1 > MAXLEN)))

tokenizer_config.json:   0%|          | 0.00/749 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/438 [00:00<?, ?B/s]

LlamaTokenizer ['<s>', '▁A', '▁yellow', '▁clock', '▁that', '▁is', '▁on', '▁the', '▁floor', '.']
captions: 5150 | token length mean 14.4 max 44 | <unk> tokens: 0
fraction truncated at 64 : 0.0


## 5. Models

In [5]:
from transformers import AutoModel, AutoImageProcessor, AutoModelForCausalLM, BitsAndBytesConfig

image_processor = AutoImageProcessor.from_pretrained(VIT_ID)
vit = AutoModel.from_pretrained(VIT_ID).to(device).eval()
for p in vit.parameters(): p.requires_grad_(False)
vit_dtype = next(vit.parameters()).dtype

llm = AutoModelForCausalLM.from_pretrained(
    LLM_PATH, quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16),
    device_map="auto", low_cpu_mem_usage=True).eval()
for p in llm.parameters(): p.requires_grad_(False)
LLM_DTYPE = next(llm.parameters()).dtype
H = llm.config.hidden_size
print("vit dtype:", vit_dtype, "| llm dtype:", LLM_DTYPE, "| hidden:", H)
assert LLM_DTYPE == torch.float16

class MLPProjector(nn.Module):
    def __init__(self, v, l):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(v, l), nn.GELU(), nn.Linear(l, l))
    def forward(self, x): return self.proj(x)

_pv = image_processor(Image.open(f"{IMG_DIR}/{val_recs[0]['image']}").convert("RGB"), return_tensors="pt")["pixel_values"]
with torch.no_grad(): _f = vit(pixel_values=_pv.to(device, dtype=vit_dtype)).last_hidden_state
print("encoder output:", tuple(_f.shape), "(expected (1, 784, 1280))")
assert tuple(_f.shape) == (1, 784, 1280)

preprocessor_config.json:   0%|          | 0.00/325 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/497 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.53GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

...

model.safetensors.index.json:   0%|          | 0.00/28.1k [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/162 [00:00<?, ?B/s]

[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


vit dtype: torch.float32 | llm dtype: torch.float16 | hidden: 4096
encoder output: (1, 784, 1280) (expected (1, 784, 1280))


## 6. Batch building, loss and evaluation helpers

In [6]:
class CapDS(Dataset):
    def __init__(self, recs): self.recs = recs
    def __len__(self): return len(self.recs)
    def __getitem__(self, i):
        r = self.recs[i]
        img = Image.open(f"{IMG_DIR}/{r['image']}").convert("RGB")
        return image_processor(img, return_tensors="pt")["pixel_values"][0], r["caption"]

def collate(b): return torch.stack([x[0] for x in b]), [x[1] for x in b]

def text_batch(caps):
    """BOS + caption + EOS, right-padded. Labels: BOS and pad positions are -100."""
    seqs = [tokenizer(c.strip()).input_ids[:MAXLEN - 1] + [tokenizer.eos_token_id] for c in caps]
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), tokenizer.pad_token_id, dtype=torch.long)
    am = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        ids[i, :len(s)] = torch.tensor(s); am[i, :len(s)] = 1
    lab = ids.clone(); lab[am == 0] = -100; lab[:, 0] = -100
    return ids.to(device), am.to(device), lab.to(device)

def run(pv, caps, P, mode="real"):
    """mode: real = own image, shuffle = image of another item in the batch, none = text only."""
    ids, am, lab = text_batch(caps)
    te = llm.get_input_embeddings()(ids)
    if mode == "none":
        emb, full_am, full_lab = te, am, lab
    else:
        with torch.no_grad():
            feats = vit(pixel_values=pv.to(device, dtype=vit_dtype)).last_hidden_state
        if mode == "shuffle": feats = feats.roll(1, dims=0)
        proj = P(feats.float())
        n_img = proj.shape[1]
        emb = torch.cat([proj, te.float()], 1)
        full_am = torch.cat([torch.ones((am.shape[0], n_img), dtype=am.dtype, device=device), am], 1)
        full_lab = torch.cat([torch.full((lab.shape[0], n_img), -100, dtype=lab.dtype, device=device), lab], 1)
    out = llm(inputs_embeds=emb.to(LLM_DTYPE), attention_mask=full_am, labels=full_lab, use_cache=False)
    return out.loss, int((full_lab[:, 1:] != -100).sum())

@torch.no_grad()
def evaluate(P, recs, mode="real", bs=4):
    """Token-weighted mean loss over recs; returns (loss, number_of_non_finite_batches)."""
    if P is not None: P.eval()
    dl = DataLoader(CapDS(recs), batch_size=bs, shuffle=False, collate_fn=collate, num_workers=NUM_WORKERS)
    tot, n, bad = 0.0, 0, 0
    for pv, caps in dl:
        loss, nt = run(pv, caps, P, mode)
        if torch.isfinite(loss): tot += float(loss) * nt; n += nt
        else: bad += 1
    if P is not None: P.train()
    return tot / max(n, 1), bad

## 7. Reference numbers (what 'good' has to beat)

In [7]:
t0 = time.time()
val_noimg, _ = evaluate(None, val_recs, "none")
_P0 = MLPProjector(1280, H).to(device)
val_rand, bad_rand = evaluate(_P0, val_recs, "real"); del _P0
print(f"val loss, NO image (text only):   {val_noimg:.4f}   <- a trained projector must get clearly BELOW this")
print(f"val loss, random-init projector:  {val_rand:.4f}  (non-finite batches: {bad_rand})")
print(f"({time.time() - t0:.0f}s)")

val loss, NO image (text only):   4.5161   <- a trained projector must get clearly BELOW this
val loss, random-init projector:  7.3136  (non-finite batches: 0)
(180s)


## 7b. Kaggle dataset for the checkpoints (created once, then one new version per push)

In [8]:
import subprocess
def sh(cmd, timeout=1800):
    r = subprocess.run(cmd, capture_output=True, text=True, timeout=timeout)
    return r.returncode, (r.stdout + r.stderr).strip()

KUSER = None
if PUSH:
    from kaggle_secrets import UserSecretsClient
    _sec = UserSecretsClient()
    os.environ["KAGGLE_USERNAME"] = _sec.get_secret("KAGGLE_USERNAME")
    os.environ["KAGGLE_KEY"] = _sec.get_secret("KAGGLE_KEY")
    try: os.environ["KAGGLE_API_TOKEN"] = _sec.get_secret("KAGGLE_API_TOKEN")      # only if you use the newer token style
    except Exception: pass
    KUSER = os.environ["KAGGLE_USERNAME"]; DS_ID = f"{KUSER}/{DATASET_SLUG}"

    with open(f"{OUT_DIR}/dataset-metadata.json", "w") as f:
        json.dump({"title": "JEPA projector fixed tokenizer ckpts", "id": DS_ID, "licenses": [{"name": "CC0-1.0"}]}, f)
    with open(f"{OUT_DIR}/README.txt", "w") as f:
        f.write("I-JEPA ViT-H/16 -> MLP projector -> frozen 4-bit Vicuna-7B-v1.5. Trained with LlamaTokenizer (verified tokens), "
                "pads/BOS masked, EOS appended. Files: best.pt, final.pt, proj_s*.pt (weights only), latest.pt (full state), log.csv, STEP.txt.\n")
    rc, out = sh(["kaggle", "datasets", "status", DS_ID], 120)
    if rc == 0:
        print("dataset exists:", DS_ID, "->", out[:80])
    else:
        rc, out = sh(["kaggle", "datasets", "create", "-p", OUT_DIR, "--dir-mode", "zip"], 600)
        print("create dataset:", rc, out[:300])
        assert rc == 0, "Could not create the Kaggle dataset (check the KAGGLE_USERNAME / KAGGLE_KEY secrets), or set PUSH = False."
    print("pushes go to:", DS_ID, "(private by default)")

def push_ckpts(note):
    """Best-effort: a failed push never stops training."""
    if not PUSH: return
    try:
        with open(f"{OUT_DIR}/STEP.txt", "w") as f:
            f.write(f"micro={state['micro']} opt={state['opt']} best_val={state['best']:.4f} note={note}\n")
        rc, out = sh(["kaggle", "datasets", "version", "-p", OUT_DIR, "-m", f"{note} micro {state['micro']}", "--dir-mode", "zip"], 1800)
        print(f"[push] {'OK' if rc == 0 else 'FAILED'} ({note}, micro {state['micro']}): {out[-200:]}")
    except Exception as e:
        print("[push] EXCEPTION, continuing:", repr(e))

dataset exists: ponishbhatia2/jepa-projector-fixed-ckpts -> Warning: Looks like you're using an outdated `kaggle` version (installed: 2.0.2)
pushes go to: ponishbhatia2/jepa-projector-fixed-ckpts (private by default)


## 8. Train

In [9]:
META = dict(tokenizer="LlamaTokenizer", tok_check=toks, transformers=have["transformers"], lr=LR, batch=BATCH,
            accum=ACCUM, eos=True, mask_pad=True, mask_bos=True, sanity=SANITY)

P = MLPProjector(1280, H).to(device)
def make_opt(): return torch.optim.AdamW(P.parameters(), lr=LR, weight_decay=WD)
opt = make_opt()

steps_per_epoch = len(train_recs) // BATCH
total_micro = EPOCHS * steps_per_epoch
total_opt = total_micro // ACCUM
def lr_at(o):
    if o < WARMUP_OPT: return LR * (o + 1) / WARMUP_OPT
    p = min(1.0, (o - WARMUP_OPT) / max(1, total_opt - WARMUP_OPT))
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * p)))
def finite_weights(m): return all(bool(torch.isfinite(p).all()) for p in m.parameters())
def snapshot(m): return {k: v.detach().cpu().clone() for k, v in m.state_dict().items()}

state = {"micro": 0, "opt": 0, "best": float("inf")}
log = []

# ---------- locate the most advanced checkpoint (local OUT_DIR or attached dataset) ----------
DATASET_SLUG = "jepa-projector-fixed-ckpts"
DATASET_DIR  = f"/kaggle/input/datasets/ponishbhatia2/{DATASET_SLUG}"
latest_path  = f"{OUT_DIR}/latest.pt"

def _opt_step(p):
    try: return torch.load(p, map_location="cpu", weights_only=False)["opt_step"]
    except Exception as e:
        print(f"unreadable {p}: {e!r}"); return -1

cands = [latest_path] if os.path.exists(latest_path) else []
cands += glob.glob(f"{DATASET_DIR}/latest.pt")
cands += glob.glob(f"/kaggle/input/**/{DATASET_SLUG}/latest.pt", recursive=True)  # fallback for other mount layouts
steps_of = {c: _opt_step(c) for c in set(cands)}
cands = sorted((c for c, s in steps_of.items() if s >= 0), key=lambda c: steps_of[c])  # most advanced last
print("checkpoint candidates (path, opt_step):", [(c, steps_of[c]) for c in cands])

src = RESUME_FROM or (cands[-1] if cands else None)
ck = torch.load(src, map_location="cpu", weights_only=False) if src else None
if ck is not None and not RESUME_FROM and ck.get("meta", {}).get("sanity") != SANITY:
    print(f"ignoring {src}: it was made with SANITY={ck.get('meta', {}).get('sanity')}, this run has SANITY={SANITY}")
    src, ck = None, None
if ck is not None:
    assert ck.get("meta", {}).get("tokenizer") == "LlamaTokenizer", "refusing to resume a checkpoint not made by this notebook (old checkpoints used the broken tokenizer)"
    P.load_state_dict(ck["projector"]); opt.load_state_dict(ck["optimizer"])
    state.update(micro=ck["step"], opt=ck["opt_step"], best=ck.get("best", float("inf")))
    log = ck.get("log", [])
    src_dir = os.path.dirname(os.path.abspath(src))
    if src_dir != os.path.abspath(OUT_DIR):
        for fn in os.listdir(src_dir):
            if not os.path.exists(f"{OUT_DIR}/{fn}"): shutil.copy(f"{src_dir}/{fn}", f"{OUT_DIR}/{fn}")
        print(f"copied earlier checkpoint files from {src_dir}")
    print(f"RESUMED from {src}: micro-step {state['micro']}, optimizer step {state['opt']}, best val {state['best']:.4f}")
elif WARM_START_FROM:
    ck = torch.load(WARM_START_FROM, map_location="cpu", weights_only=False); sd = ck["projector"] if "projector" in ck else ck
    P.load_state_dict(sd); print("WARM START (weights only) from", WARM_START_FROM, "- optimizer and counters fresh")
else:
    print("NO CHECKPOINT FOUND - starting from scratch")

def save_ckpt(path, full=True):
    obj = {"projector": snapshot(P), "step": state["micro"], "opt_step": state["opt"], "best": state["best"], "meta": META}
    if full: obj["optimizer"] = opt.state_dict(); obj["log"] = log
    tmp = path + ".tmp"; torch.save(obj, tmp); os.replace(tmp, path)

last_good = snapshot(P)
def write_log():
    with open(f"{OUT_DIR}/log.csv", "w") as f:
        f.write("micro,opt,train_loss,val_loss,val_nonfinite,lr,seconds\n")
        for r in log: f.write(",".join(str(r[k]) for k in ("micro", "opt", "train_loss", "val_loss", "val_nonfinite", "lr", "seconds")) + "\n")

print(f"steps/epoch {steps_per_epoch} micro | total {total_micro} micro = {total_opt} optimizer steps | warmup {WARMUP_OPT}")
push_ckpts("start")
t0 = time.time(); t_last = t0
run_loss, run_n, bad_streak, skipped, last_train = 0.0, 0, 0, 0, float("nan")
start_micro = state["micro"]; P.train(); opt.zero_grad(set_to_none=True)
try:
    for ep in range(start_micro // steps_per_epoch, EPOCHS):
        g = torch.Generator().manual_seed(SEED + ep)
        order = torch.randperm(len(train_recs), generator=g).tolist()[: steps_per_epoch * BATCH]
        skip = max(0, start_micro - ep * steps_per_epoch)
        idx = order[skip * BATCH:]
        dl = DataLoader(CapDS([train_recs[i] for i in idx]), batch_size=BATCH, shuffle=False,
                        collate_fn=collate, num_workers=NUM_WORKERS, drop_last=True)
        for pv, caps in dl:
            loss, _ = run(pv, caps, P, "real")
            state["micro"] += 1
            if not torch.isfinite(loss):
                bad_streak += 1; skipped += 1
                opt.zero_grad(set_to_none=True)
                print(f"[skip] micro {state['micro']}: non-finite loss ({bad_streak}/{MAX_BAD_STREAK} in a row)")
                if bad_streak >= MAX_BAD_STREAK: raise RuntimeError("too many consecutive non-finite batches")
                continue
            bad_streak = 0
            (loss / ACCUM).backward()
            run_loss += float(loss); run_n += 1

            stepped = False
            if state["micro"] % ACCUM == 0:
                for gp in opt.param_groups: gp["lr"] = lr_at(state["opt"])
                gn = torch.nn.utils.clip_grad_norm_(P.parameters(), CLIP)
                if not torch.isfinite(gn):
                    opt.zero_grad(set_to_none=True); skipped += 1
                    print(f"[skip] micro {state['micro']}: non-finite grad norm")
                else:
                    opt.step(); opt.zero_grad(set_to_none=True); state["opt"] += 1; stepped = True
                    if finite_weights(P):
                        last_good = snapshot(P)
                    else:
                        print(f"[rollback] micro {state['micro']}: weights went non-finite, restoring last good")
                        P.load_state_dict(last_good); opt = make_opt(); stepped = False

            if state["micro"] % 50 == 0 and run_n:
                now = time.time(); per = (now - t_last) / 50; t_last = now
                eta = per * (total_micro - state["micro"]) / 60
                last_train = run_loss / run_n
                print(f"micro {state['micro']:>5}/{total_micro} opt {state['opt']:>4} | train {run_loss / run_n:.4f} | lr {opt.param_groups[0]['lr']:.2e} | {per:.2f}s/micro | ETA {eta:.0f} min | skipped {skipped}")
                run_loss, run_n = 0.0, 0

            if stepped and state["opt"] % VAL_EVERY == 0:
                v, vb = evaluate(P, val_recs)
                log.append(dict(micro=state["micro"], opt=state["opt"], train_loss=round(last_train, 4),
                                val_loss=round(v, 4), val_nonfinite=vb, lr=opt.param_groups[0]["lr"], seconds=int(time.time() - t0)))
                write_log()
                tag = ""
                if vb == 0 and v < state["best"]:
                    state["best"] = v; save_ckpt(f"{OUT_DIR}/best.pt", full=False); tag = "  <- best"
                print(f"   VAL @ opt {state['opt']}: {v:.4f}  (no-image {val_noimg:.4f}, gain {val_noimg - v:+.3f}) | non-finite val batches {vb}{tag}")
            if stepped and state["opt"] % SAVE_EVERY == 0: save_ckpt(latest_path)
            if stepped and state["opt"] % MILESTONE_EVERY == 0: save_ckpt(f"{OUT_DIR}/proj_s{state['micro']}.pt", full=False)
            if stepped and PUSH and state["opt"] % PUSH_EVERY == 0: save_ckpt(latest_path); push_ckpts("periodic")
except BaseException as e:
    print("STOPPED:", repr(e))
    state["micro"] -= state["micro"] % ACCUM
    if finite_weights(P): save_ckpt(latest_path); print("latest.pt saved; rerun this cell to resume"); push_ckpts("stopped")
    raise
save_ckpt(latest_path); save_ckpt(f"{OUT_DIR}/final.pt", full=False); push_ckpts("final")
print(f"done in {(time.time() - t0) / 60:.1f} min | skipped batches: {skipped} | best val {state['best']:.4f}")

checkpoint candidates (path, opt_step): [('/kaggle/input/datasets/ponishbhatia2/jepa-projector-fixed-ckpts/latest.pt', 214)]
copied earlier checkpoint files from /kaggle/input/datasets/ponishbhatia2/jepa-projector-fixed-ckpts
RESUMED from /kaggle/input/datasets/ponishbhatia2/jepa-projector-fixed-ckpts/latest.pt: micro-step 1712, optimizer step 214, best val 2.0164
steps/epoch 2500 micro | total 7500 micro = 937 optimizer steps | warmup 40
[push] OK (start, micro 1712): , 144MB/s]
100%|█████████▉| 251M/252M [00:01<00:00, 150MB/s]
100%|██████████| 252M/252M [00:02<00:00, 125MB/s]

  0%|          | 0.00/239 [00:00<?, ?B/s]
100%|██████████| 239/239 [00:00<00:00, 859B/s]


/tmp/ipykernel_23/3827643668.py:95: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  run_loss += float(loss); run_n += 1


micro  1750/7500 opt  218 | train 2.1138 | lr 1.83e-04 | 3.07s/micro | ETA 294 min | skipped 0
micro  1800/7500 opt  225 | train 2.0407 | lr 1.82e-04 | 4.06s/micro | ETA 386 min | skipped 0
micro  1850/7500 opt  231 | train 2.0082 | lr 1.81e-04 | 4.06s/micro | ETA 382 min | skipped 0
micro  1900/7500 opt  237 | train 1.9808 | lr 1.80e-04 | 4.10s/micro | ETA 383 min | skipped 0
micro  1950/7500 opt  243 | train 2.0210 | lr 1.78e-04 | 4.10s/micro | ETA 379 min | skipped 0
micro  2000/7500 opt  250 | train 2.1188 | lr 1.77e-04 | 4.09s/micro | ETA 375 min | skipped 0
   VAL @ opt 250: 1.9886  (no-image 4.5161, gain +2.528) | non-finite val batches 0  <- best
micro  2050/7500 opt  256 | train 2.1461 | lr 1.76e-04 | 7.68s/micro | ETA 697 min | skipped 0
micro  2100/7500 opt  262 | train 2.0568 | lr 1.74e-04 | 4.09s/micro | ETA 368 min | skipped 0
micro  2150/7500 opt  268 | train 2.0159 | lr 1.73e-04 | 4.08s/micro | ETA 364 min | skipped 0
micro  2200/7500 opt  275 | train 2.0491 | lr 1.71e-

## 9. Final checks: real vs shuffled vs no-image, curves, sample captions

In [10]:
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
Pe = MLPProjector(1280, H).to(device)
rows = {}
for name in ("best", "final"):
    path = f"{OUT_DIR}/{name}.pt"
    if not os.path.exists(path): continue
    Pe.load_state_dict(torch.load(path, map_location="cpu")["projector"])
    real, b1 = evaluate(Pe, val_recs, "real"); shuf, b2 = evaluate(Pe, val_recs, "shuffle")
    trn, b3 = evaluate(Pe, train_recs[:100], "real")
    rows[name] = (real, shuf, trn)
    print(f"[{name}] val real {real:.4f} | val shuffled-image {shuf:.4f} (gap {shuf - real:+.3f}) | no-image {val_noimg:.4f} (gain {val_noimg - real:+.3f}) | train100 {trn:.4f} | non-finite batches {b1 + b2 + b3}")

if log:
    xs = [r["opt"] for r in log]
    plt.figure(figsize=(6, 3.5)); plt.plot(xs, [r["val_loss"] for r in log], marker="o", label="val")
    plt.axhline(val_noimg, ls="--", color="gray", label="no-image val"); plt.xlabel("optimizer step"); plt.ylabel("loss")
    plt.legend(); plt.tight_layout(); plt.savefig(f"{OUT_DIR}/curve.png", dpi=120); plt.show()

@torch.no_grad()
def caption(P_, rec, max_new=32):
    pv = CapDS([rec])[0][0].unsqueeze(0)
    feats = vit(pixel_values=pv.to(device, dtype=vit_dtype)).last_hidden_state
    bos = torch.tensor([[tokenizer.bos_token_id]], device=device)
    emb = torch.cat([P_(feats.float()), llm.get_input_embeddings()(bos).float()], 1).to(LLM_DTYPE)
    out = llm.generate(inputs_embeds=emb, attention_mask=torch.ones(emb.shape[:2], dtype=torch.long, device=device),
                       max_new_tokens=max_new, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0], skip_special_tokens=True)

if "best" in rows or "final" in rows:
    Pe.load_state_dict(torch.load(f"{OUT_DIR}/{'best' if 'best' in rows else 'final'}.pt", map_location="cpu")["projector"])
    for r in val_recs[:6]:
        try: print(f"{r['image'][-16:-4]} | GT: {r['caption'].strip()}\n   gen: {caption(Pe, r)}")
        except Exception as e: print("generation failed:", repr(e)); break
print("\nFiles in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))

[best] val real 1.8704 | val shuffled-image 2.8637 (gap +0.993) | no-image 4.5161 (gain +2.646) | train100 1.5367 | non-finite batches 0
[final] val real 1.8942 | val shuffled-image 3.1295 (gap +1.235) | no-image 4.5161 (gain +2.622) | train100 1.3556 | non-finite batches 0


[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


000000109753 | GT: A yellow clock that is on the floor.
   gen: A plate of food sitting on a table.


[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


000000230008 | GT: a traffic motorcycle cop waits to give a ticket
   gen: A man riding a motorcycle with a sidecar full of people.


...

000000015567 | GT: The cat is standing by the rocking chair in the living room.
   gen: A black and white cat sitting on a bed.


[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


000000384718 | GT: An elephant is standing outside of the water while the others are inside.
   gen: A group of elephants swimming in the ocean.


[transformers] Both `max_new_tokens` (=32) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


000000122745 | GT: A stop sign is lit up in the dark of night.
   gen: A man is standing in front of a stop sign.

Files in /kaggle/working/ckpt_fixed : ['README.txt', 'STEP.txt', 'best.pt', 'curve.png', 'dataset-metadata.json', 'final.pt', 'latest.pt', 'log.csv', 'proj_s1600.pt', 'proj_s2400.pt', 'proj_s3200.pt', 'proj_s4000.pt', 'proj_s4800.pt', 'proj_s5600.pt', 'proj_s6400.pt', 'proj_s7200.pt', 'proj_s800.pt']


## What to look at (and what to send back)

**Sanity run (`SANITY = True`)**: the VAL lines should fall steadily and end clearly **below the no-image value** printed in section 7, with `gain` positive and growing, no `[skip]`/`[rollback]` lines, and section 9 showing a positive shuffled-image gap. Captions after only ~100 steps will be rough; they should at least be fluent English (not `<unk>` fragments).

**Full run (`SANITY = False`)**: about 937 optimizer steps (3 epochs). Use *Save Version -> Save & Run All* so it keeps running with the tab closed. `latest.pt` is written every 50 optimizer steps and pushed to the checkpoint dataset every 100, with one push at the start to confirm pushing works. If the session dies, add `<your-username>/jepa-projector-fixed-ckpts` as an input and run all cells: the training cell finds `latest.pt`, copies the earlier checkpoint files across and continues from the same data order. A checkpoint made with a different `SANITY` value is ignored.

**Outputs** (in `/kaggle/working/ckpt_fixed/`, also pushed to the private Kaggle dataset `<your-username>/jepa-projector-fixed-ckpts` every 100 optimizer steps and at the end; section 7b creates it and needs the `KAGGLE_USERNAME` and `KAGGLE_KEY` secrets under Add-ons -> Secrets): `best.pt` (lowest clean val), `final.pt`, `proj_s*.pt` milestones, `latest.pt` (full state), `log.csv`, `curve.png`. Checkpoints keep the old format (`{"projector": ..., "step": ...}`) plus a `meta` block, so the analysis notebook can load them (it must also use `LlamaTokenizer`).

**Optional warm-start experiment**: set `WARM_START_FROM` to an old checkpoint (weights only) to see whether it speeds things up. Treat it as an experiment, because it gives that run extra effective training steps.